In [1]:
!pip install langchain-text-splitters chromadb sentence-transformers groq pypdf tavily-python pandas openpyxl python-telegram-bot nest_asyncio -q
print("✅ All packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.4/745.4 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq
from tavily import TavilyClient
import pandas as pd
import json
import os

# ── API KEYS ──────────────────────────────────────────
GROQ_API_KEY = "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"
TAVILY_API_KEY = "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"
TELEGRAM_TOKEN = "8675318472:AAF6z7Flf80DMrYCCE_8F642-YxPrC9r3Kg"

client = Groq(api_key=GROQ_API_KEY)
tavily = TavilyClient(api_key=TAVILY_API_KEY)

# ── LOAD PDFs ─────────────────────────────────────────
def load_pdf(filepath):
    reader = PdfReader(filepath)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

print("📄 Loading documents...")
safety = load_pdf("safety_manual.pdf")
maintenance = load_pdf("equipment_maintenance.pdf")
outage = load_pdf("outage_procedures.pdf")

# ── CHUNK DOCUMENTS ───────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
safety_chunks = splitter.split_text(safety)
maintenance_chunks = splitter.split_text(maintenance)
outage_chunks = splitter.split_text(outage)

# ── LOAD EXCEL ────────────────────────────────────────
print("📊 Loading procurement data...")
history_df = pd.read_excel("procurement_data.xlsx", sheet_name="Procurement_History")
suppliers_df = pd.read_excel("procurement_data.xlsx", sheet_name="Suppliers")
pending_df = pd.read_excel("procurement_data.xlsx", sheet_name="Pending_Requirements")

history_chunks = splitter.split_text(history_df.to_string(index=False))
suppliers_chunks = splitter.split_text(suppliers_df.to_string(index=False))
pending_chunks = splitter.split_text(pending_df.to_string(index=False))

# ── CREATE KNOWLEDGE BASE ─────────────────────────────
print("🧠 Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("power_company_complete")

all_chunks = []
all_ids = []
all_metadata = []

for i, chunk in enumerate(safety_chunks):
    all_chunks.append(chunk)
    all_ids.append(f"safety_{i}")
    all_metadata.append({"source": "safety_manual"})

for i, chunk in enumerate(maintenance_chunks):
    all_chunks.append(chunk)
    all_ids.append(f"maintenance_{i}")
    all_metadata.append({"source": "equipment_maintenance"})

for i, chunk in enumerate(outage_chunks):
    all_chunks.append(chunk)
    all_ids.append(f"outage_{i}")
    all_metadata.append({"source": "outage_procedures"})

for i, chunk in enumerate(history_chunks):
    all_chunks.append(chunk)
    all_ids.append(f"history_{i}")
    all_metadata.append({"source": "procurement_history"})

for i, chunk in enumerate(suppliers_chunks):
    all_chunks.append(chunk)
    all_ids.append(f"suppliers_{i}")
    all_metadata.append({"source": "suppliers"})

for i, chunk in enumerate(pending_chunks):
    all_chunks.append(chunk)
    all_ids.append(f"pending_{i}")
    all_metadata.append({"source": "pending_requirements"})

embeddings = embedding_model.encode(all_chunks).tolist()
collection.add(
    documents=all_chunks,
    embeddings=embeddings,
    ids=all_ids,
    metadatas=all_metadata
)

print(f"✅ Knowledge base ready with {len(all_chunks)} chunks!")

📄 Loading documents...
📊 Loading procurement data...
🧠 Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Knowledge base ready with 38 chunks!


In [6]:
# ── SEARCH FUNCTIONS ──────────────────────────────────
def search_knowledge_base(query):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=4)
    chunks = results['documents'][0]
    sources = [m['source'] for m in results['metadatas'][0]]
    output = ""
    for chunk, source in zip(chunks, sources):
        output += f"[From {source}]:\n{chunk}\n\n"
    return output

def search_web(query):
    results = tavily.search(query=query + " India INR price 2026", max_results=3)
    output = ""
    for r in results["results"]:
        output += f"Title: {r['title']}\nContent: {r['content']}\n\n"
    return output

# ── RESEARCHER AGENT ──────────────────────────────────
def researcher_agent(question):
    print("  [🔍 Researcher searching...]")
    kb_results = search_knowledge_base(question)
    web_results = search_web(question)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a researcher for a power company in Hyderabad, India.
Compile relevant findings from company knowledge base and web search.
Include specific numbers, dates and facts. Always mention sources.

CRITICAL RULES:
- When question involves prices — ALWAYS include actual price numbers in INR from web search
- Never tell user to search themselves — YOU find the prices and give them directly
- If web search has current prices — include them with source
- Always complete full research before responding"""
            },
            {
                "role": "user",
                "content": f"""Question: {question}

Company Knowledge Base:
{kb_results}

Web Search Results:
{web_results}

Compile all findings into a clear research summary with actual numbers."""
            }
        ]
    )
    research = response.choices[0].message.content
    print("  [✅ Researcher done!]")
    return research

# ── ANALYST AGENT ─────────────────────────────────────
def analyst_agent(question, research):
    print("  [📊 Analyst analyzing...]")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a senior procurement analyst for a power company in Hyderabad, India.
Analyze research findings and give practical recommendations.
Be specific, use actual numbers in INR, and be actionable.

Format your response clearly:
1. Key Findings
2. Analysis
3. Recommendations
4. Action Items"""
            },
            {
                "role": "user",
                "content": f"""Question: {question}

Research Findings:
{research}

Provide professional analysis and recommendations."""
            }
        ]
    )
    analysis = response.choices[0].message.content
    print("  [✅ Analyst done!]")
    return analysis

print("✅ Multi Agent System ready!")

✅ Multi Agent System ready!


In [8]:
import asyncio
import nest_asyncio
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes

nest_asyncio.apply()

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    welcome = """👋 Welcome to Power Company AI Assistant!

I can help you with:
⚡ Safety procedures and PPE requirements
🔧 Equipment maintenance guides
📦 Procurement history and prices
🏭 Supplier information
📋 Pending requirements
🌐 Current market prices

Just ask me anything in plain English!

Example questions:
• What are our pending requirements?
• What is current market price of transformer oil?
• Which supplier has best rating?
• What PPE is required for 33KV work?
• Compare our last purchase price with current market"""

    await update.message.reply_text(welcome)

async def help_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "🆘 Just type your question in plain English!\n\n"
        "I search through:\n"
        "📚 Safety manuals\n"
        "🔧 Maintenance guides\n"
        "📦 Procurement history\n"
        "🏭 Supplier database\n"
        "🌐 Live market prices\n\n"
        "No special commands needed — just ask!"
    )

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    question = update.message.text
    print(f"\n👤 Question: {question}")

    await context.bot.send_chat_action(
        chat_id=update.effective_chat.id,
        action="typing"
    )
    await update.message.reply_text("🔍 Researching your question, please wait...")

    try:
        research = researcher_agent(question)
        analysis = analyst_agent(question, research)

        if "MEMORY_UPDATE:" in analysis:
            analysis = analysis.split("MEMORY_UPDATE:")[0].strip()

        if len(analysis) > 4000:
            analysis = analysis[:4000] + "\n\n_(Response truncated)_"

        await update.message.reply_text(analysis)
        print("✅ Answered successfully!")

    except Exception as e:
        print(f"❌ Error: {e}")
        await update.message.reply_text(
            "❌ Sorry, I encountered an error. Please try again!"
        )

async def main():
    app = ApplicationBuilder().token(TELEGRAM_TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("help", help_command))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))

    print("\n✅ AI Bot is running!")

    print("They can search it on Telegram and start chatting!\n")

    await app.initialize()
    await app.start()
    await app.updater.start_polling()
    await asyncio.sleep(3600)

asyncio.run(main())


✅ AI Bot is running!
They can search it on Telegram and start chatting!



ERROR:telegram.ext.Updater:Exception happened while polling for updates.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_utils/networkloop.py", line 161, in network_retry_loop
    await do_action()
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_utils/networkloop.py", line 154, in do_action
    action_cb_task.result()
  File "/usr/lib/python3.12/asyncio/futures.py", line 202, in result
    raise self._exception.with_traceback(self._exception_tb)
  File "/usr/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_updater.py", line 340, in polling_action_cb
    updates = await self.bot.get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_extbot.py", line 672, in get_updates
    updates = await super().get_updates(
          


👤 Question: Whats the transformer oil price now?
  [🔍 Researcher searching...]
  [✅ Researcher done!]
  [📊 Analyst analyzing...]


ERROR:telegram.ext.Updater:Exception happened while polling for updates.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_utils/networkloop.py", line 161, in network_retry_loop
    await do_action()
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_utils/networkloop.py", line 154, in do_action
    action_cb_task.result()
  File "/usr/lib/python3.12/asyncio/futures.py", line 202, in result
    raise self._exception.with_traceback(self._exception_tb)
  File "/usr/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_updater.py", line 340, in polling_action_cb
    updates = await self.bot.get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_extbot.py", line 672, in get_updates
    updates = await super().get_updates(
          

  [✅ Analyst done!]
✅ Answered successfully!


ERROR:telegram.ext.Updater:Exception happened while polling for updates.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_utils/networkloop.py", line 161, in network_retry_loop
    await do_action()
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_utils/networkloop.py", line 154, in do_action
    action_cb_task.result()
  File "/usr/lib/python3.12/asyncio/futures.py", line 202, in result
    raise self._exception.with_traceback(self._exception_tb)
  File "/usr/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_updater.py", line 340, in polling_action_cb
    updates = await self.bot.get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_extbot.py", line 672, in get_updates
    updates = await super().get_updates(
          

KeyboardInterrupt: 